In [1]:
%%capture
!pip install evaluate
!pip install datasets
!pip install "transformers==4.57.2"
!pip install sentencepiece
!pip install emoji
!pip install "accelerate>=0.26.0"

In [2]:
import os
import re
import shutil

import emoji
import evaluate
import numpy as np
import pandas as pd
import torch
from datasets import Dataset, DatasetDict
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    EarlyStoppingCallback,
    Trainer,
    TrainingArguments,
)

In [3]:
def clean_text(text):
    if pd.isna(text):
        return text

    # 1. lowercase
    text = text.lower()

    # 2. remove @USER mentions
    text = re.sub(r"@user", "", text, flags=re.IGNORECASE)
    text = re.sub(r"@url", "", text, flags=re.IGNORECASE)

    # 3. remove URLs (actual links or placeholder "URL")
    # text = re.sub(r'http\S+|https\S+|url', '', text, flags=re.IGNORECASE)

    # 4. remove underscores, repeated underscores
    text = re.sub(r"_+", " ", text)

    # 5. remove slashes
    text = text.replace("\\", " ").replace("/", " ")

    # 6. remove emojis
    text = emoji.replace_emoji(text, replace="")

    # 7. remove quotation marks (normal + smart)
    text = re.sub(r"[\"“”]", "", text)

    # 8. normalize spaces
    text = re.sub(r"\s+", " ", text).strip()

    return text

In [4]:
# Setup
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [18]:
%%time
# Load Model & Tokenizer
MODEL_NAME = "cardiffnlp/twitter-roberta-base-hate-latest"

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
base_model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME, num_labels=2
)

base_model.to(device)

tokenizer_config.json:   0%|          | 0.00/351 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/280 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/888 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

CPU times: user 1.36 s, sys: 853 ms, total: 2.21 s
Wall time: 5.56 s


RobertaForSequenceClassification(
  (roberta): RobertaModel(
    (embeddings): RobertaEmbeddings(
      (word_embeddings): Embedding(50265, 768, padding_idx=1)
      (position_embeddings): Embedding(514, 768, padding_idx=1)
      (token_type_embeddings): Embedding(1, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): RobertaEncoder(
      (layer): ModuleList(
        (0-11): 12 x RobertaLayer(
          (attention): RobertaAttention(
            (self): RobertaSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): RobertaSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
         

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

In [19]:
# Load Data
DATA_DIR = "data"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
DEV_DIR = os.path.join(DATA_DIR, "dev")
TEST_DIR = os.path.join(DATA_DIR, "test")


def load_split(split_dir):
    dfs = []
    if not os.path.exists(split_dir):
        print(f"Directory not found: {split_dir}")
        return pd.DataFrame()
    for file in os.listdir(split_dir):
        if file.endswith(".csv"):
            lang = file.replace(".csv", "")
            df = pd.read_csv(os.path.join(split_dir, file))
            df = df[["text", "polarization"]]
            df["lang"] = lang
            dfs.append(df)
    if not dfs:
        return pd.DataFrame()
    return pd.concat(dfs, ignore_index=True)


print("Loading Train Data...")
raw_train_df = load_split(TRAIN_DIR)
print(f"Loaded {len(raw_train_df)} training examples")

print("Loading Dev Data (Used as internal Test)...")
raw_dev_df = load_split(DEV_DIR)
print(f"Loaded {len(raw_dev_df)} dev examples")

print("Loading Test Data (For Submission)...")
raw_test_df = load_split(TEST_DIR)
print(f"Loaded {len(raw_test_df)} test examples")

Loading Train Data...
Loaded 73681 training examples
Loading Dev Data (Used as internal Test)...
Loaded 3687 dev examples
Loading Test Data (For Submission)...
Loaded 33288 test examples


In [20]:
# Preprocess Text
print("Preprocessing text (cleaning)...")
raw_train_df["text"] = raw_train_df["text"].astype(str).apply(clean_text)
raw_dev_df["text"] = raw_dev_df["text"].astype(str).apply(clean_text)
raw_test_df["text"] = raw_test_df["text"].astype(str).apply(clean_text)

Preprocessing text (cleaning)...


In [21]:
# Data Splitting
# Rename 'polarization' to 'labels'
if "polarization" in raw_train_df.columns:
    raw_train_df = raw_train_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_dev_df.columns:
    raw_dev_df = raw_dev_df.rename(columns={"polarization": "labels"})
if "polarization" in raw_test_df.columns:
    raw_test_df = raw_test_df.rename(columns={"polarization": "labels"})

In [22]:
# Create Dataset Objects
train_dataset = Dataset.from_pandas(raw_train_df[["text", "labels"]], preserve_index=False)
val_dataset = Dataset.from_pandas(raw_dev_df[["text", "labels"]], preserve_index=False)
test_dataset = Dataset.from_pandas(raw_test_df[["text", "labels"]], preserve_index=False)

dataset = DatasetDict(
    {"train": train_dataset, "validation": val_dataset, "test": test_dataset}
)

In [23]:
pd.DataFrame(train_dataset[:5]).head()

,text,labels
0,ఒత్తిడిని ఒంటరిగా భరించాల్సిన అవసరం లేదు. ఎవరై...,0
1,సైనికుల కుటుంబాలు వారు విధులలో ఉన్నప్పుడు అపార...,0
2,ఒక వ్యక్తి యొక్క లింగ గుర్తింపును గౌరవించకుండా...,1
3,వివిధ దేశాల సంస్కృతుల గురించి తెలుసుకోవడం ద్వా...,1
4,రాజకీయ రంగంలో ప్రజాస్వామ్య విలువలను కాపాడే బాధ...,0


In [24]:
%%time


# Tokenize
def tokenize_function(examples):
    return tokenizer(
        examples["text"], padding="max_length", truncation=True, max_length=256
    )


print("Tokenizing datasets...")
encoded_dataset = dataset.map(tokenize_function, batched=True)
encoded_dataset.set_format(
    type="torch", columns=["input_ids", "attention_mask", "labels"]
)

Tokenizing datasets...


Map:   0%|          | 0/73681 [00:00<?, ? examples/s]

Map:   0%|          | 0/3687 [00:00<?, ? examples/s]

Map:   0%|          | 0/33288 [00:00<?, ? examples/s]

CPU times: user 19.4 s, sys: 254 ms, total: 19.6 s
Wall time: 8.59 s


In [25]:
# Training Arguments
OUTPUT_DIR = "./output_results"
BATCH_SIZE = 32
EPOCHS = 50
LR = 2e-5
GRAD_ACCUM = 2

# Calculate steps
steps_per_epoch = len(encoded_dataset["train"]) // (BATCH_SIZE * GRAD_ACCUM)
eval_steps = steps_per_epoch
steps_per_epoch

1151

In [26]:
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    learning_rate=LR,
    weight_decay=0.01,
    warmup_steps=1000,
    gradient_accumulation_steps=GRAD_ACCUM,
    logging_steps=eval_steps,
    eval_steps=eval_steps,
    save_steps=eval_steps * 60,  # Save less frequently to save space
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    eval_strategy="steps",
    logging_dir=f"{OUTPUT_DIR}/logs",
    report_to="none",
    fp16=torch.cuda.is_available(),
)

metric_f1 = evaluate.load("f1")
metric_acc = evaluate.load("accuracy")


def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=1)
    f1 = metric_f1.compute(predictions=predictions, references=labels, average="macro")[
        "f1"
    ]
    acc = metric_acc.compute(predictions=predictions, references=labels)["accuracy"]
    return {"f1": f1, "accuracy": acc}


trainer = Trainer(
    model=base_model,
    args=training_args,
    train_dataset=encoded_dataset["train"],
    eval_dataset=encoded_dataset["validation"],
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
)

In [27]:
%%time
# Start Training
trainer.train()

Step,Training Loss,Validation Loss,F1,Accuracy
1151,0.656700,0.565380,0.697446,0.698400
2302,0.533100,0.533665,0.730882,0.730947
3453,0.494800,0.547642,0.716914,0.723895
4604,0.457000,0.541932,0.748296,0.749390
5755,0.424600,0.548018,0.752157,0.754001
6906,0.393200,0.594689,0.750576,0.754001
8057,0.364700,0.584110,0.757583,0.758883
9208,0.336300,0.675970,0.753624,0.754272
10359,0.315500,0.623861,0.757004,0.759425
11510,0.295700,0.736720,0.761687,0.762680


CPU times: user 31min 10s, sys: 33.3 s, total: 31min 43s
Wall time: 31min 42s


TrainOutput(global_step=14963, training_loss=0.38940549408266956, metrics={'train_runtime': 1902.1327, 'train_samples_per_second': 1936.8, 'train_steps_per_second': 30.282, 'total_flos': 1.2590758576564224e+17, 'train_loss': 0.38940549408266956, 'epoch': 12.989144594007815})

In [28]:
%%time
# Evaluation on Internal Test Set (Dev Folder)
print("Evaluating on Internal Test Set (Dev folder data)...")
preds_output = trainer.predict(encoded_dataset["test"])

pred_labels = np.argmax(preds_output.predictions, axis=1)
true_labels = preds_output.label_ids

print("\nClassification Report:")
report = classification_report(
    true_labels, pred_labels, target_names=["Not Polar (0)", "Polar (1)"], digits=4
)
print(f"\n{report}")

macro_f1 = f1_score(true_labels, pred_labels, average="macro")
print(f"Macro F1: {macro_f1:.4f}")

# Per-Language Analysis
raw_test_df["preds"] = pred_labels
print("\n=== Macro F1 per Language ===")
results = []
for lang in sorted(raw_test_df["lang"].unique()):
    lang_df = raw_test_df[raw_test_df["lang"] == lang]
    f1 = f1_score(lang_df["labels"], lang_df["preds"], average="macro")
    acc = accuracy_score(lang_df["labels"], lang_df["preds"])
    print(f"{lang}: F1={f1:.4f}, Acc={acc:.4f}, Support={len(lang_df)}")
    results.append(
        {"lang": lang, "f1_macro": f1, "accuracy": acc, "count": len(lang_df)}
    )

results_df = pd.DataFrame(results)
print(f"\nAverage Macro F1 across languages: {results_df['f1_macro'].mean():.4f}")

Evaluating on Internal Test Set (Dev folder data)...



Classification Report:

               precision    recall  f1-score   support

Not Polar (0)     0.7430    0.7315    0.7372     15562
    Polar (1)     0.7674    0.7779    0.7726     17726

     accuracy                         0.7562     33288
    macro avg     0.7552    0.7547    0.7549     33288
 weighted avg     0.7560    0.7562    0.7561     33288

Macro F1: 0.7549

=== Macro F1 per Language ===
amh: F1=0.5644, Acc=0.7382, Support=1501
arb: F1=0.6935, Acc=0.6936, Support=1521
ben: F1=0.7226, Acc=0.7308, Support=1501
deu: F1=0.6553, Acc=0.6578, Support=1432
eng: F1=0.7932, Acc=0.8079, Support=1452
fas: F1=0.7269, Acc=0.8208, Support=1484
hau: F1=0.6729, Acc=0.8899, Support=1644
hin: F1=0.7217, Acc=0.8762, Support=1236
ita: F1=0.5546, Acc=0.5839, Support=1538
khm: F1=0.5470, Acc=0.9076, Support=2988
mya: F1=0.7679, Acc=0.7679, Support=1301
nep: F1=0.7907, Acc=0.7907, Support=903
ori: F1=0.5848, Acc=0.7176, Support=1066
pan: F1=0.6502, Acc=0.6564, Support=809
pol: F1=0.6778, Acc=0.